In [17]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import seaborn 

In [2]:
fss_path = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/Intercomparison/FSS/ensemble"
out_dir = "/home/users/mendrika/Zambia-Intercomparison/comparison/fss/thresholded"
os.makedirs(out_dir, exist_ok=True)

lead_times = [1, 2, 4, 6]

In [10]:
files = [f for f in os.listdir(fss_path) if f.endswith("_by_threshold.csv")]
pattern = r"fss_hour_(\d{2})_t(\d)_by_threshold\.csv"

In [ ]:
all_dfs = []
for fname in files:
    m = re.match(pattern, fname)
    if not m:
        continue
    hour = int(m.group(1))
    lt = int(m.group(2))
    df = pd.read_csv(os.path.join(fss_path, fname))
    df["hour"] = hour
    df["lead_time"] = lt

    # round scale to nearest 10 km
    if "scale_km" in df.columns:
        df["scale_km"] = (df["scale_km"] / 10).round() * 10

    all_dfs.append(df)

if not all_dfs:
    raise RuntimeError("No Fraction Skill Score CSV files found. Check path or pattern.")

fss_all = pd.concat(all_dfs, ignore_index=True)

In [12]:
fss_all

,window,scale_km,threshold,zcast,nflics,netncc,hour,lead_time
0,3,10.0,0.1,0.217584,0.002420,0.285281,0,1
1,9,30.0,0.1,0.250508,0.002877,0.328257,0,1
2,25,80.0,0.1,0.328369,0.004848,0.467838,0,1
3,49,150.0,0.1,0.383133,0.008184,0.690444,0,1
4,81,240.0,0.1,0.359350,0.012218,0.846618,0,1
...,...,...,...,...,...,...,...,...
2183035,9,30.0,0.9,0.000005,0.000005,0.000005,23,6
2183036,25,80.0,0.9,0.000016,0.000016,0.000016,23,6
2183037,49,150.0,0.9,0.000055,0.000055,0.000055,23,6
2183038,81,240.0,0.9,0.000141,0.000141,0.000141,23,6


In [18]:
import seaborn as sns


# ---------------------------------------------------------------------
# Prepare data (averaged across all hours)
# ---------------------------------------------------------------------
df_mean = (
    fss_all.groupby(["lead_time", "threshold", "scale_km"], as_index=False)[["zcast", "netncc"]]
    .mean()
)

# Ensure thresholds and scales are sorted
df_mean = df_mean.sort_values(["lead_time", "threshold", "scale_km"])

# ---------------------------------------------------------------------
# Plot heatmaps: FSS (NetNCC vs ZCAST) as function of scale × threshold
# ---------------------------------------------------------------------
for lt in sorted(df_mean["lead_time"].unique()):
    df_lt = df_mean[df_mean["lead_time"] == lt]

    # Pivot to get threshold × scale matrices
    pivot_netncc = df_lt.pivot(index="threshold", columns="scale_km", values="netncc")
    pivot_zcast = df_lt.pivot(index="threshold", columns="scale_km", values="zcast")

    # Create side-by-side subplots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    vmin, vmax = 0, 1  # fixed scale for consistency

    sns.heatmap(pivot_netncc, ax=axes[0], cmap="viridis", vmin=vmin, vmax=vmax, cbar=False)
    sns.heatmap(pivot_zcast, ax=axes[1], cmap="viridis", vmin=vmin, vmax=vmax, cbar=True)

    # Titles and labels
    axes[0].set_title(f"NetNCC — Lead Time {lt} h", fontsize=12, weight="bold")
    axes[1].set_title(f"ZCAST — Lead Time {lt} h", fontsize=12, weight="bold")

    for ax in axes:
        ax.set_xlabel("Spatial scale (km)")
        ax.set_ylabel("Threshold")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    out_file = os.path.join(out_dir, f"fss_heatmap_scale_threshold_t{lt}_avg_hour.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_file}")


Saved: /home/users/mendrika/Zambia-Intercomparison/comparison/fss/thresholded/fss_heatmap_scale_threshold_t1_avg_hour.png
Saved: /home/users/mendrika/Zambia-Intercomparison/comparison/fss/thresholded/fss_heatmap_scale_threshold_t2_avg_hour.png
Saved: /home/users/mendrika/Zambia-Intercomparison/comparison/fss/thresholded/fss_heatmap_scale_threshold_t4_avg_hour.png
Saved: /home/users/mendrika/Zambia-Intercomparison/comparison/fss/thresholded/fss_heatmap_scale_threshold_t6_avg_hour.png
